# 03 — RAG prototype

**Purpose:** prototype the Career Mentor chain end to end and inspect the
retrieved context alongside each answer.

- Index `data/career_notes/`
- Wire retriever -> prompt -> LLM (this is exactly `src/mentor/rag_chain.py`)
- Inspect retrieved chunks alongside each answer
- Confirm the mentor refuses / says "I don't know" on out-of-scope questions

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.mentor.rag_chain import build_mentor_index, build_mentor_retriever, ask_mentor

index = build_mentor_index(force_rebuild=True)
print(f'Mentor index built with {index.index.ntotal} chunk(s).')

In [ ]:
# Inspect raw retrieval before it goes into the prompt.
retriever = build_mentor_retriever(k=3)
question = 'How do I switch into a Data Analyst role from an unrelated background?'
docs = retriever.invoke(question)
for d in docs:
    print('---', d.metadata.get('filename'), '---')
    print(d.page_content[:300], '...\n')

In [ ]:
# Full pipeline: retrieve -> prompt -> LLM -> guardrails -> grounding check.
questions = [
    'How do I switch into a Data Analyst role from an unrelated background?',
    'What should I focus on to prepare for a software engineering interview?',
    'How much does a senior ML engineer earn in Tokyo in 2026?',  # not in our docs -> should refuse
    'What is the capital of France?',  # off-topic -> should decline
]

for q in questions:
    result = ask_mentor(q)
    print('Q:', q)
    print('A:', result['answer'][:400])
    print('sources:', result['sources'], '| grounded:', result['grounded'], '| blocked:', result['blocked'])
    print()

### Hallucination check

The third question (senior ML engineer salary in Tokyo) is a *plausible-sounding* career
question that is nonetheless **not answerable from `data/career_notes/`** (no salary tables
exist in the corpus). A correct mentor should say it doesn't have that information rather
than inventing a number. The fourth question is off-topic entirely and should be declined
on scope. Both are scored `grounded: True` by `guardrails.is_grounded()` because an honest
refusal counts as grounded behaviour, not a failure — this is the hallucination check the
spec (section 8) asks for, and both results feed directly into `reports/answer_quality.md`
via `src/evaluate.py`.

In [ ]:
# Conversation memory (stretch goal): pass prior turns as history.
history = []
q1 = 'I want to move from a support role into software engineering. Where do I start?'
r1 = ask_mentor(q1, history=history)
print('Q1:', q1)
print('A1:', r1['answer'][:400])
history.append({'question': q1, 'answer': r1['answer']})

q2 = 'What about the interview process specifically?'
r2 = ask_mentor(q2, history=history)
print('\nQ2:', q2)
print('A2:', r2['answer'][:400])